In [1]:
!pip -q install -U "transformers>=4.44" accelerate

In [2]:
!pip -q install sentence_transformers --upgrade

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:64"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["WANDB_DISABLED"] = "true"


In [12]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Mixed-batch SentenceTransformers training with strict per-batch composition:
  [1] summary -> doc
  [2] doc -> doc (German noisy)
  [3] doc -> doc (French noisy)
  [4] summary -> summary_noise
  [5] query -> doc
  [6] query -> summary
  [7] TED q -> q (de <-> fr)
  [8] LUX q -> q (lb <-> {de,fr,en})

Batch size must be 8 (one sample from each pool per step).
"""

import os
import re
import json
import random
import itertools
from typing import List, Dict, Tuple, Iterable
import argparse
import shutil
import gc

import numpy as np
import pandas as pd
import torch
from torch.utils.data import IterableDataset, DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses



def set_random_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_dataset_local(file_path: str) -> List[dict]:
    """Loads a dataset (list of JSON objects) from a JSONL file."""
    dataset = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            dataset.append(json.loads(line))
    return dataset

def clean_text_from_punctuation(text: str) -> str:
    return re.sub(r'[^a-zA-Z0-9\s]+', ' ', str(text)).strip()

def extract_parallel_sentences(lines: List[dict], src_col: str, tgt_col: str, min_chars: int = 5) -> List[Tuple[str, str]]:
    """Extract (src, tgt) sentence pairs from JSONL records that have a 'translation' list of dicts."""
    parallel_sentences = []
    for data in lines:
        translations = data.get('translation', [])
        for sentence_pair in translations:
            if not isinstance(sentence_pair, dict):
                continue
            src_sentence = sentence_pair.get(src_col)
            tgt_sentence = sentence_pair.get(tgt_col)
            if isinstance(src_sentence, list) or isinstance(tgt_sentence, list):
                continue
            if src_sentence and tgt_sentence:
                s = clean_text_from_punctuation(src_sentence)
                t = clean_text_from_punctuation(tgt_sentence)
                if len(s) >= min_chars and len(t) >= min_chars:
                    parallel_sentences.append((src_sentence, tgt_sentence))
    return parallel_sentences

def to_examples_from_pairs(pairs: List[Tuple[str, str]], left_prefix: str = "", right_prefix: str = "") -> List[InputExample]:

    try:
      result = [InputExample(texts=[left_prefix + a, right_prefix + b], label=1.0) for a, b in pairs]
    except TypeError as type_error:
      print(f'this is the type error {type_error}')
      print(pairs)
    return [InputExample(texts=[left_prefix + a, right_prefix + b], label=1.0) for a, b in pairs]

def sample_pairs(pairs: List[Tuple[str, str]], k: int = None, seed: int = 42) -> List[Tuple[str, str]]:
    rng = random.Random(seed)
    if k is not None and k < len(pairs):
        return rng.sample(pairs, k)
    pairs = pairs[:]
    rng.shuffle(pairs)
    return pairs


def build_pools(
    summary_df: pd.DataFrame,        # columns: doc_text, summary, query, doc_text_noised, summary_noised, query_noised
    mono_df: pd.DataFrame,
    doc_df_de: pd.DataFrame,
    doc_df_fr: pd.DataFrame,
    lux_pairs: List[Tuple[str, str]],
    model_name: str
) -> Dict[str, List[InputExample]]:
    """
    Create eight pools of InputExample, one per relation type.
    Applies e5/gte-style prefixes automatically for e5/gte models.
    """
    name_l = model_name.lower()
    is_prompt_model = ("multilingual-e5" in name_l) or ("intfloat/e5" in name_l) or ("gte" in name_l)
    q_prefix = "query: " if is_prompt_model else ""
    p_prefix = "passage: " if is_prompt_model else ""

    pools = {k: [] for k in [
        "sum2doc", "dd_german", "dd_french", "sum2sum_noise",
        "q2doc", "q2sum", "ted_q2q", "lux_q2q"
    ]}

    # 1) summary -> doc
    if {"summary", "text_noised"}.issubset(summary_df.columns):
        print('summary -> doc')
        pairs = list(zip(summary_df["summary"], summary_df["text_noised"]))
        pools["sum2doc"] = to_examples_from_pairs(pairs, q_prefix, p_prefix)

    # 2) doc -> doc (German noisy)
    if {"text", "text_noised"}.issubset(doc_df_de.columns):
        print('doc -> doc (German noisy)')
        pairs = list(zip(doc_df_de["text"], doc_df_de["text_noised"]))
        pools["dd_german"] = to_examples_from_pairs(pairs, p_prefix, p_prefix)

    # 3) doc -> doc (French noisy)
    if {"text", "text_noised"}.issubset(doc_df_fr.columns):
        print('doc -> doc (French noisy)')

        pairs = list(zip(doc_df_fr["text"], doc_df_fr["text_noised"]))
        pools["dd_french"] = to_examples_from_pairs(pairs, p_prefix, p_prefix)

    # 4) summary -> summary_noise
    if {"summary", "summary_noised"}.issubset(summary_df.columns):
        print('summary -> summary_noise')

        pairs = list(zip(summary_df["summary"], summary_df["summary_noised"]))
        pools["sum2sum_noise"] = to_examples_from_pairs(pairs, q_prefix, p_prefix)

    # 5) q -> doc
    if {"query", "text_noised"}.issubset(summary_df.columns):
        print('q -> doc)')

        pairs = list(zip(summary_df["query"], summary_df["text_noised"]))
        pools["q2doc"] = to_examples_from_pairs(pairs, q_prefix, p_prefix)

    # 6) q -> summary
    if {"query", "summary_noised"}.issubset(summary_df.columns):
        print('q -> summary')

        pairs = list(zip(summary_df["query"], summary_df["summary_noised"]))
        pools["q2sum"] = to_examples_from_pairs(pairs, q_prefix, p_prefix)

    # 7) TED q -> q (using de<->fr as cross-lingual queries)
    if {"sentence", "sentence_noise_random05"}.issubset(mono_df.columns):
        print('TED q -> q')

        pairs = list(zip(mono_df["sentence"], mono_df["sentence_noise_random05"]))
        pools["ted_q2q"] = to_examples_from_pairs(pairs, q_prefix, q_prefix)

    # 8) Lux q -> q (lb<->{de,fr,en})
    print('Lux q -> q')

    pools["lux_q2q"] = to_examples_from_pairs(lux_pairs, q_prefix, q_prefix)

    # Finalize: prune empties and shuffle
    for k, lst in list(pools.items()):
        pools[k] = [ex for ex in lst if ex.texts[0].strip() and ex.texts[1].strip()]
        random.shuffle(pools[k])
    return pools


class MixedBatchIterableDataset(IterableDataset):
    """
    Yields items in a fixed per-batch pattern of 8 example types:
        [sum2doc, dd_langA, dd_langB, sum2sum_noise, q2doc, q2sum, ted_q2q, lux_q2q]
    One from each pool per step to guarantee the mix.
    """
    def __init__(self, pools: Dict[str, List[InputExample]], steps_per_epoch: int, seed: int = 42):
        super().__init__()
        self.pools = pools
        self.steps = int(steps_per_epoch)
        self.seed = seed
        self.spec = ["sum2doc", "dd_german", "dd_french", "sum2sum_noise", "q2doc", "q2sum", "ted_q2q", "lux_q2q"]
        for key in self.spec:
            if key not in self.pools or len(self.pools[key]) == 0:
                raise ValueError(f"Pool '{key}' is missing or empty. Check your inputs for {key}.")

    def _epoch_iterators(self):
        # Shuffle each pool reproducibly; cycle if pool shorter than steps
        rng = random.Random(self.seed)
        iters = {}
        for k, lst in self.pools.items():
            pool = lst[:]  # copy
            rng.shuffle(pool)
            if len(pool) < self.steps:
                iters[k] = itertools.cycle(pool)
            else:
                iters[k] = iter(pool)
        return iters

    def __iter__(self) -> Iterable[InputExample]:
        iters = self._epoch_iterators()
        produced = 0
        target = self.steps * 8
        while produced < target:
            for key in self.spec:
                try:
                    item = next(iters[key])
                except StopIteration:
                    pool = self.pools[key][:]
                    random.shuffle(pool)
                    iters[key] = iter(pool)
                    item = next(iters[key])
                yield item
                produced += 1
                if produced >= target:
                    break

def train_mixed(
    model_name: str,
    summary_df: pd.DataFrame,
    mono_df: pd.DataFrame,
    doc_df_de: pd.DataFrame,
    doc_df_fr: pd.DataFrame,
    lux_pairs: List[Tuple[str, str]],
    batch_size: int = 8,
    steps_per_epoch: int = 10_000,
    epochs: int = 1,
    seed: int = 42,
    out_dir: str = "model-out"
) -> str:
    assert batch_size == 8, "This pipeline enforces batch_size=8 (one of each type per batch)."
    set_random_seed(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    pools = build_pools(summary_df, mono_df,doc_df_de,doc_df_fr, lux_pairs, model_name)

    keys = ["sum2doc", "dd_german", "dd_french", "sum2sum_noise", "q2doc", "q2sum", "ted_q2q", "lux_q2q"]
    for k in keys:
        print(f"[pool {k}] size = {len(pools[k])}")
    min_pool = min(len(pools[k]) for k in keys)
    if steps_per_epoch > min_pool:
        print(f"[warn] steps_per_epoch ({steps_per_epoch}) > smallest pool ({min_pool}). That pool will cycle/repeat.")

    dataset = MixedBatchIterableDataset(pools, steps_per_epoch=steps_per_epoch, seed=seed)

    model = SentenceTransformer(model_name, trust_remote_code=True).to(device)

    loss = losses.MultipleNegativesRankingLoss(model)
    warmup = int(0.1 * steps_per_epoch)

    train_loader = DataLoader(dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=model.smart_batching_collate,
        pin_memory=False,
        num_workers=0,
        drop_last=True
    )

    model.fit(
        train_objectives=[(train_loader, loss)],
        epochs=epochs,
        warmup_steps=warmup,
        show_progress_bar=True,
        use_amp=True
    )

    os.makedirs(out_dir, exist_ok=True)
    save_path = os.path.join(out_dir, f"{model_name.replace('/','_')}-mixed8-seed{seed}")
    model.save(save_path)
    print("Model saved to:", save_path)
    return save_path


def parse_seeds(seeds_str: str) -> List[int]:
    return [int(s.strip()) for s in seeds_str.split(",") if s.strip()]

def main():
    parser = argparse.ArgumentParser(description="Mixed-batch (8-type) SentenceTransformers training.")
    parser.add_argument("--model_name", type=str, default="Alibaba-NLP/gte-multilingual-base")
    parser.add_argument("--epochs", type=int, default=1)
    parser.add_argument("--batch_size", type=int, default=8, help="Must be 8 (one from each pool).")
    parser.add_argument("--steps_per_epoch", type=int, default=10_000, help="Number of steps per epoch.")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--seeds", type=str, default="42,100,123,777,999", help="Comma-separated list of seeds.")
    parser.add_argument("--out_dir", type=str, default="model-out")
    parser.add_argument("--document_file_de", type=str, default="./finetuning_data/de_docs_random_noise.csv",
                help="CSV with columns: text, text_noised")
    parser.add_argument("--document_file_fr", type=str, default="./finetuning_data/fr_docs_random_noise.csv",
                help="CSV with columns: text, text_noised")
    parser.add_argument("--summary_file", type=str, default="./finetuning_data/query_doc_dataset_random_noise.csv",
                    help="CSV with columns: doc_text, summary, query, doc_text_noised, summary_noised, query_noised")
    parser.add_argument("--mono", type=str, default="./finetuning_data/TED_data_random_noise_concat.csv",
                        help="CSV with german,german_noise,french,french_noise")
    parser.add_argument("--lux_file_en", type=str, default="./finetuning_data/lb_en_training_set.jsonl")
    parser.add_argument("--lux_file_de", type=str, default="./finetuning_data/lb_de_training_set.jsonl")
    parser.add_argument("--lux_file_fr", type=str, default="./finetuning_data/lb_fr_training_set.jsonl")

    parser.add_argument("--lux_sample_k", type=int, default=20160,
                        help="Pairs sampled per lb->{de,fr,en} before concat. Use -1 for all.")

    args, _unknown = parser.parse_known_args()

    set_random_seed(args.seed)

    doc_df_de = pd.read_csv(args.document_file_de)
    doc_df_fr = pd.read_csv(args.document_file_fr)
    summary_df = pd.read_csv(args.summary_file)
    mono_df = pd.read_csv(args.mono)

    lb_de_training_set_loaded = load_dataset_local(args.lux_file_de)
    lb_fr_training_set_loaded = load_dataset_local(args.lux_file_fr)
    lb_en_training_set_loaded = load_dataset_local(args.lux_file_en)

    lb_de_training_sentences = extract_parallel_sentences(lb_de_training_set_loaded, src_col="lb", tgt_col="de")
    lb_fr_training_sentences = extract_parallel_sentences(lb_fr_training_set_loaded, src_col="lb", tgt_col="fr")
    lb_en_training_sentences = extract_parallel_sentences(lb_en_training_set_loaded, src_col="lb", tgt_col="en")

    print(f"Loaded LUX (articles): de={len(lb_de_training_set_loaded)} | fr={len(lb_fr_training_set_loaded)} | en={len(lb_en_training_set_loaded)}")
    print(f"Extracted pairs: de={len(lb_de_training_sentences)} | fr={len(lb_fr_training_sentences)} | en={len(lb_en_training_sentences)}")

    k = None if args.lux_sample_k == -1 else args.lux_sample_k
    de_sampled = sample_pairs(lb_de_training_sentences, k, seed=args.seed)
    fr_sampled = sample_pairs(lb_fr_training_sentences, k, seed=args.seed)
    en_sampled = sample_pairs(lb_en_training_sentences, k, seed=args.seed)

    lux_pairs = de_sampled + fr_sampled + en_sampled
    lux_pairs = list({(a.strip(), b.strip()) for (a, b) in lux_pairs})
    random.Random(args.seed).shuffle(lux_pairs)
    print(f"Lux combined pool size (deduped): {len(lux_pairs)}")

    seeds = parse_seeds(args.seeds)
    saved = []
    for s in seeds:
        try:
            print(f"\n===== Seed {s} =====")
            set_random_seed(s)
            mono_df = mono_df.sample(frac=1, random_state=s).reset_index(drop=True)
            doc_df_de = doc_df_de.sample(frac=1, random_state=s).reset_index(drop=True)
            doc_df_fr = doc_df_fr.sample(frac=1, random_state=s).reset_index(drop=True)
            summary_df = summary_df.sample(frac=1, random_state=s).reset_index(drop=True)

            save_path = train_mixed(
                model_name=args.model_name,
                summary_df=summary_df,
                mono_df=mono_df,
                doc_df_de = doc_df_de,
                doc_df_fr = doc_df_fr,
                lux_pairs=lux_pairs,
                batch_size=args.batch_size,
                steps_per_epoch=args.steps_per_epoch,
                epochs=args.epochs,
                seed=s,
                out_dir=args.out_dir
            )
            saved.append(save_path)

            base_name = save_path
            root_dir = os.path.dirname(save_path) or "."
            base_dir = os.path.basename(save_path)
            zip_path = shutil.make_archive(base_name=base_name, format="zip", root_dir=root_dir, base_dir=base_dir)
            print("Zipped:", zip_path)

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except torch.cuda.OutOfMemoryError as e:
            print(f"[OOM at seed {s}] Consider lowering sequence/window size or batch_size. Error: {e}")
            raise
        except Exception as e:
            print(f"[Error at seed {s}] {e}")
            raise

    print("\nSaved model folders:")
    for p in saved:
        print(" -", p)





In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
main()

Using device: cuda
Loaded LUX (articles): de=2105 | fr=2105 | en=2105
Extracted pairs: de=20113 | fr=20040 | en=19518
Lux combined pool size (deduped): 58616

===== Seed 42 =====
Using device: cuda
summary -> doc
doc -> doc (German noisy)
doc -> doc (French noisy)
summary -> summary_noise
q -> doc)
q -> summary
TED q -> q
Lux q -> q
[pool sum2doc] size = 10000
[pool dd_german] size = 10000
[pool dd_french] size = 10000
[pool sum2sum_noise] size = 10000
[pool q2doc] size = 10000
[pool q2sum] size = 10000
[pool ted_q2q] size = 348658
[pool lux_q2q] size = 58616


Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


[OOM at seed 42] Consider lowering sequence/window size or batch_size. Error: CUDA out of memory. Tried to allocate 734.00 MiB. GPU 0 has a total capacity of 39.56 GiB of which 320.88 MiB is free. Process 3354 has 39.23 GiB memory in use. Of the allocated memory 38.53 GiB is allocated by PyTorch, and 205.27 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


OutOfMemoryError: CUDA out of memory. Tried to allocate 734.00 MiB. GPU 0 has a total capacity of 39.56 GiB of which 320.88 MiB is free. Process 3354 has 39.23 GiB memory in use. Of the allocated memory 38.53 GiB is allocated by PyTorch, and 205.27 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)